# Fairness audit of in-context learning: results analysis

This notebook reproduces every quantity the manuscript reports, from the raw
prediction files under `results/` to the three audit levels and the
fairness--utility frontier.

**The audit has three levels**, each stricter than the last:

1. **Raw GF** — a cell is *group-fair* when it is jointly compliant on
   equal-opportunity difference (`|EOD| <= 0.10`) and disparate impact
   (`0.80 <= DI <= 1.20`).
2. **Non-collapsed GF** — raw GF that additionally survives the collapse
   filter: a cell whose classifier answers one label almost always, or whose
   group rates saturate, is fair only because it has stopped discriminating
   between instances at all.
3. **Non-collapsed GF + utility** — the cell must also classify at least as
   well as the majority-class predictor, scored with the same
   group-unweighted estimator.

Neither is a normative claim of fairness: the gates are *descriptors* used to
audit a protocol, not a validated fairness standard.

A **cell** is one (condition, model, dataset, protected attribute, test split)
combination. All metric definitions come from `analysis/cells.py`, which is the
single source of truth shared with the table generators, so a number cannot be
computed two ways.

## 1. Setup

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd() / "analysis"))
import cells as C

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

FIGURES = Path("analysis") / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

print(f"results: {C.RESULTS}")
print(f"bands:   |EOD| <= {C.EOD_BAND} and DI in [{C.DI_BAND[0]}, {C.DI_BAND[1]}]")
print(f"collapse: PPP extreme margin {C.SUPPORT_MARGIN}, FPR saturation band {C.COLLAPSE_FPR_MARGIN}")

## 2. Load the cells

One row per (condition, model, dataset, protected attribute, test split),
carrying the fairness metrics, the per-group rates, the collapse flag and the
per-pass accuracy.

In [ ]:
cells = C.load_cells()
flat = cells.reset_index()

print(f"{len(flat)} cells | {flat['cond'].nunique()} conditions | "
      f"{flat['model'].nunique()} models | {flat['dataset'].nunique()} datasets")
print(f"raw GF {int(flat['raw_gf'].sum())} | non-collapsed GF {int(flat['gf'].sum())} | "
      f"collapsed {int(flat['collapsed'].sum())}")

coverage = flat.groupby(["cond", "test"]).size().unstack(fill_value=0)
coverage

### Which conditions are scored on both splits?

The manuscript reports a condition on both canonical test splits only when the
runner has produced both. `DUAL_TEST` in `cells.py` is that list.

In [ ]:
AVAILABLE = {(c, t) for c, t in zip(flat["cond"], flat["test"])}
DUAL_AVAILABLE = sorted({c for c in flat["cond"].unique()
                         if (c, "D1") in AVAILABLE and (c, "D2") in AVAILABLE})

def tests_of(cond):
    """Splits actually present for a condition, in canonical order."""
    return [t for t in ("D1", "D2") if (cond, t) in AVAILABLE]

print(f"scored on both splits ({len(DUAL_AVAILABLE)}): {', '.join(DUAL_AVAILABLE)}")

## 3. The three audit levels

The headline of the paper is the gap between the levels: how much apparent
fairness is an artefact of a classifier that has collapsed, and how much of
what remains belongs to a classifier not worth deploying.

In [ ]:
def trivial_baselines():
    """Majority-class accuracy per (dataset, feature, test), group-unweighted.

    Scored with the same estimator as the cell accuracy -- the unweighted mean
    over the two protected groups -- so the comparison is like for like.
    """
    out = {}
    for (dataset, test), data in C.load_predictions().items():
        y = data["y_true"]
        majority = int(y.mean() >= 0.5)
        for feature, sensitive in data["sensitive"].items():
            per_group = [(y[sensitive == g] == majority).mean()
                         for g in (1, 0) if (sensitive == g).any()]
            out[(dataset, feature, test)] = float(np.mean(per_group))
    return out

baselines = trivial_baselines()
flat["baseline"] = [baselines[(r.dataset, r.feature, r.test)] for r in flat.itertuples()]
flat["useful"] = flat["acc"] >= flat["baseline"]
flat["margin"] = flat["acc"] - flat["baseline"]
flat["gf_useful"] = flat["gf"] & flat["useful"]

levels = pd.DataFrame({
    "cells": flat.groupby("test").size(),
    "raw GF": flat.groupby("test")["raw_gf"].sum(),
    "non-collapsed GF": flat.groupby("test")["gf"].sum(),
    "+ utility": flat.groupby("test")["gf_useful"].sum(),
})
levels["collapse removes"] = levels["raw GF"] - levels["non-collapsed GF"]
levels["utility removes"] = levels["non-collapsed GF"] - levels["+ utility"]
levels

In [ ]:
for test in sorted(flat["test"].unique()):
    part = flat[flat["test"] == test]
    raw, gf, useful = int(part["raw_gf"].sum()), int(part["gf"].sum()), int(part["gf_useful"].sum())
    print(f"[T{test[-1]}] {len(part)} cells: raw GF {raw} -> non-collapsed {gf} "
          f"(-{raw - gf}) -> useful {useful} (-{gf - useful}, "
          f"{C.pct(gf - useful, gf)}% of the non-collapsed GF cells)")
    print(f"       mean margin over the trivial baseline: "
          f"NC-GF cells {part[part['gf']]['margin'].mean():+.3f}, "
          f"others {part[~part['gf']]['margin'].mean():+.3f}")

### Does the utility gate reorder the interventions?

If the ranking of conditions by non-collapsed GF survives the utility gate, the
third level is a refinement. If it does not, any recommendation read off the
second level is unsafe.

In [ ]:
for test in sorted(flat["test"].unique()):
    part = flat[flat["test"] == test]
    by_cond = part.groupby("cond").agg(gf=("gf", "sum"), gf_useful=("gf_useful", "sum"),
                                       n=("gf", "size"))
    by_cond = by_cond.sort_values("gf", ascending=False)
    rank_gf = list(by_cond.index)
    rank_useful = list(by_cond.sort_values("gf_useful", ascending=False).index)
    print(f"--- T{test[-1]} ---")
    print(f"leader by NC-GF: {rank_gf[0]} | by NC-GF+utility: {rank_useful[0]} | "
          f"top-3 changes: {'yes' if rank_gf[:3] != rank_useful[:3] else 'no'}")
    display(by_cond)

## 4. Collapse: what the second gate removes

A collapsed cell is one whose predicted-positive rate sits at an extreme, or
whose group rates saturate. Such a cell can pass both fairness bands while
being useless as a classifier, which is why raw GF alone overstates fairness.

In [ ]:
collapse_by_cond = flat.pivot_table(index="cond", columns="test",
                                    values=["raw_gf", "gf", "collapsed"], aggfunc="sum")
collapse_by_cond = collapse_by_cond.reindex([c for c in C.CONDITIONS if c in flat["cond"].values])
collapse_by_cond

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, level in zip(axes, ["collapsed", "gf"]):
    pivot = (flat[flat["test"] == "D1"]
             .pivot_table(index="cond", columns="model", values=level, aggfunc="sum")
             .reindex(index=[c for c in C.CONDITIONS if c in flat["cond"].values],
                      columns=[m for m in C.MODEL_ORDER if m in flat["model"].values]))
    # D4r runs on a subset of the models, so some (condition, model) pairs are empty.
    absent = pivot.isna().to_numpy()
    pivot = pivot.fillna(0)
    im = ax.imshow(pivot.to_numpy(), cmap="magma_r" if level == "collapsed" else "viridis",
                   aspect="auto", vmin=0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([C.COND_LABEL[c].replace("\\_", "_") for c in pivot.index])
    ax.set_title("Collapsed cells (T1)" if level == "collapsed" else "Non-collapsed GF cells (T1)")
    ax.grid(False)
    values = pivot.to_numpy()
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            label = "." if absent[i, j] else f"{int(values[i, j])}"
            ax.text(j, i, label, ha="center", va="center", fontsize=7,
                    color="white" if values[i, j] > values.max() * 0.6 else "black")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle("Where fairness comes from: collapse against non-collapsed GF", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "collapse_vs_gf.png", bbox_inches="tight")
plt.show()

## 5. Transfer between regimes

Conditions scored on both canonical splits can be read across regimes: T1 is
the original-distribution test (historical label) and T2 the
corrected-distribution one. $\Delta_{\text{GF}}$ is the change in
non-collapsed-GF cells from T1 to T2.

The two regimes are **not commensurable**: the utility readings are defined
against different references and no instance carries both labels. Each is read
on its own; the delta measures transfer, not improvement.

In [ ]:
if not DUAL_AVAILABLE:
    print("[skip] no condition has both splits yet.")
else:
    rows = []
    for cond in DUAL_AVAILABLE:
        d1 = flat[(flat["cond"] == cond) & (flat["test"] == "D1")]
        d2 = flat[(flat["cond"] == cond) & (flat["test"] == "D2")]
        rows.append({
            "cond": cond,
            "n": len(d1),
            "GF T1": int(d1["gf"].sum()),
            "GF T2": int(d2["gf"].sum()),
            "delta (cells)": int(d2["gf"].sum() - d1["gf"].sum()),
            "delta (pp)": C.pct(int(d2["gf"].sum() - d1["gf"].sum()), len(d1)),
            "acc T1": d1["acc"].mean(),
            "acc T2": d2["acc"].mean(),
        })
    transfer = pd.DataFrame(rows).set_index("cond")
    display(transfer)
    print("\n[warning] a difference of one to three cells is within resampling noise;")
    print("          read the bootstrap intervals in section 6 before claiming an ordering.")

## 6. Bootstrap intervals

Test instances are resampled once per (dataset, test split) and shared by every
cell scored on it, then the non-collapsed-GF count is recounted on each
replicate with `cells.genuine_matrix`, which mirrors `src/metrics.py` exactly.

The intervals cover **sampling variability of the test instances only**, not
the variability a different sample of demonstrations would induce.

`REPLICATES` is set low so the notebook runs end to end in a few minutes; the
manuscript uses 2000.

In [ ]:
REPLICATES = 200
SEED = 42

def replicate_counts(preds, replicates, seed):
    """Non-collapsed-GF count per (cond, test) on the sample (row 0) and replicates."""
    rng = np.random.default_rng(seed)
    stacked = {key: (list(g["preds"]), np.vstack([g["preds"][k] for k in g["preds"]]))
               for key, g in preds.items()}
    counts = {}
    for r in range(replicates + 1):
        for (dataset, test), (keys, matrix) in stacked.items():
            group = preds[(dataset, test)]
            n = len(group["y_true"])
            idx = None if r == 0 else rng.integers(0, n, n)
            for feature in C.PROTECTED[dataset]:
                _, _, _, genuine = C.genuine_matrix(
                    group["y_true"], group["sensitive"][feature], 1, matrix, idx)
                for (cond, _), value in zip(keys, genuine):
                    counts.setdefault((cond, test), np.zeros(replicates + 1, dtype=int))[r] += int(value)
    return counts

preds = C.load_predictions()
counts = replicate_counts(preds, REPLICATES, SEED)
print(f"{REPLICATES} replicates over {len(preds)} (dataset, test) groups")

In [ ]:
def interval(samples):
    return int(np.percentile(samples, 2.5)), int(np.percentile(samples, 97.5))

rows = []
for cond in C.CONDITIONS:
    for test in tests_of(cond):
        c = counts[(cond, test)]
        lo, hi = interval(c[1:])
        rows.append({"cond": cond, "test": f"T{test[-1]}", "GF": int(c[0]),
                     "lo": lo, "hi": hi, "CI": f"[{lo}, {hi}]"})
boot = pd.DataFrame(rows)
display(boot.set_index(["cond", "test"]))

In [ ]:
if not DUAL_AVAILABLE:
    print("[skip] delta intervals need both splits.")
else:
    rows = []
    for cond in DUAL_AVAILABLE:
        n = len(flat[(flat["cond"] == cond) & (flat["test"] == "D1")])
        delta = counts[(cond, "D2")] - counts[(cond, "D1")]
        lo, hi = interval(delta[1:])
        rows.append({"cond": cond, "delta (pp)": C.pct(int(delta[0]), n),
                     "lo": C.pct(lo, n), "hi": C.pct(hi, n),
                     "crosses zero": "yes" if lo <= 0 <= hi else "no"})
    deltas = pd.DataFrame(rows).set_index("cond")
    display(deltas)
    crossing = list(deltas.index[deltas["crosses zero"] == "yes"])
    print(f"\nconditions whose delta interval covers zero: {', '.join(crossing) or 'none'}")

### Do the orderings between conditions hold?

A gap between two conditions is the difference of their counts on the *same*
resample, so the comparison is paired. "Ordering held" is the share of
replicates in which the first condition stayed ahead.

In [ ]:
GAPS = [("D4b_D2", "D4b_D1", "D2"), ("D4a", "D4b_D2", "D2"), ("D4a", "D4b_D1", "D2"),
        ("D4b_D1", "D6_fairprompt", "D1"), ("D6_fairprompt", "D1_original", "D1"),
        ("D5_decontam", "D1_original", "D1"), ("D2_fair_causal", "D1_original", "D1"),
        ("D4r_D2", "D4r_D1", "D2"), ("D4r_D2", "D4r_0", "D2")]

rows = []
for first, second, test in GAPS:
    if (first, test) not in counts or (second, test) not in counts:
        continue
    gap = counts[(first, test)] - counts[(second, test)]
    lo, hi = interval(gap[1:])
    rows.append({"pair": f"{first} - {second}", "test": f"T{test[-1]}",
                 "gap": int(gap[0]), "CI": f"[{lo:+d}, {hi:+d}]",
                 "ordering held": f"{100 * float(np.mean(gap[1:] > 0)):.1f}%"})
pd.DataFrame(rows).set_index(["pair", "test"]) if rows else print("[skip] no pair available yet.")

## 7. Sensitivity of the collapse criterion

The collapse criterion has two margins: the saturation band on the
false-positive rate (0.10 in the paper) and the predicted-positive extreme
(0.01). Both are varied here and the counts recomputed from the stored
predictions, so the table answers whether the conclusions depend on where the
cut sits.

In [ ]:
FPR_MARGINS = [0.15, 0.10, 0.05]
SUPPORT_MARGINS = [0.01, 0.05]

def recount(preds, fpr_margin, support_margin):
    out = {}
    for (dataset, test), group in preds.items():
        keys = list(group["preds"])
        matrix = np.vstack([group["preds"][k] for k in keys])
        for feature in C.PROTECTED[dataset]:
            _, _, _, genuine = C.genuine_matrix(
                group["y_true"], group["sensitive"][feature], 1, matrix,
                fpr_margin=fpr_margin, support_margin=support_margin)
            for (cond, _), value in zip(keys, genuine):
                out[(cond, test)] = out.get((cond, test), 0) + int(value)
    return out

variants = [(f, s) for s in SUPPORT_MARGINS for f in FPR_MARGINS]
table = {v: recount(preds, *v) for v in variants}

published = {(c, t): int(flat[(flat["cond"] == c) & (flat["test"] == t)]["gf"].sum())
             for c in C.CONDITIONS for t in tests_of(c)}
mismatch = {k: (table[(0.10, 0.01)][k], published[k])
            for k in published if table[(0.10, 0.01)][k] != published[k]}
assert not mismatch, f"recount at the published thresholds disagrees with the tables: {mismatch}"
print("recount at the published thresholds reproduces the tables exactly.")

rows = []
for (cond, test), n_published in published.items():
    counts_v = [table[v][(cond, test)] for v in variants]
    rows.append({"cond": cond, "test": f"T{test[-1]}", "published": n_published,
                 **{f"m={f:.2f}/s={s:.2f}": c for (f, s), c in zip(variants, counts_v)},
                 "max shift": max(abs(c - n_published) for c in counts_v)})
sensitivity = pd.DataFrame(rows).set_index(["cond", "test"])
print(f"largest shift across the six settings: {sensitivity['max shift'].max()} cells")
sensitivity

## 8. Retrainable baselines (XGBoost)

The retrainable paradigm on the same canonical splits, covering all three
mitigation families: pre-processing (training on D1, D2 (FLAI) and D3
resampling), in-processing (the reduction with an equalized-odds constraint)
and post-processing (per-group thresholds). Same metrics, same gates, same
collapse flag as the LLM conditions.

In [ ]:
try:
    base = C.load_baselines()
except (ValueError, KeyError) as exc:
    base = None
    print(f"[skip] no baseline outputs under results/*/baselines yet ({exc}).")

if base is not None:
    summary = (base.groupby(["baseline", "test"])
               .agg(cells=("gf", "size"), gf=("gf", "sum"),
                    collapsed=("collapsed", "sum"), acc=("acc", "mean")))
    display(summary)
    print("\nnon-collapsed GF over cells (3 datasets x 2 attributes per split)")

## 9. Fairness--utility frontier

Within a regime the utility reading is single-valued, so a frontier is
constructible: conditions are placed on (label agreement, non-collapsed GF) and
the non-dominated set reported. A frontier *across* regimes is not identifiable,
so each split is read on its own.

In [ ]:
from scipy import stats

def frontier(points):
    """Indices of the non-dominated conditions on (accuracy, non-collapsed GF)."""
    keep = []
    for i, p in enumerate(points):
        if not any(q[0] >= p[0] and q[1] >= p[1] and (q != p).any() for q in points):
            keep.append(i)
    return keep

fig, axes = plt.subplots(1, len(sorted(flat["test"].unique())), figsize=(11, 4.2), squeeze=False)
for ax, test in zip(axes[0], sorted(flat["test"].unique())):
    frame = (flat[flat["test"] == test].groupby("cond")
             .agg(gf=("gf", "mean"), acc=("acc", "mean")))
    frame["gf"] *= 100
    keep = frontier(frame[["acc", "gf"]].to_numpy())
    on = frame.index[keep]

    r, p = stats.pearsonr(frame["acc"], frame["gf"])
    ax.scatter(frame["acc"], frame["gf"], s=34, c="#b0b0b0", zorder=2, label="dominated")
    ax.scatter(frame.loc[on, "acc"], frame.loc[on, "gf"], s=54, c="#1f77b4",
               zorder=3, label="non-dominated")
    # Conditions that land on nearly the same point get their labels stacked.
    placed = []
    for cond in frame.index:
        x, y = frame.loc[cond, "acc"], frame.loc[cond, "gf"]
        span = frame["gf"].max() - frame["gf"].min()
        drop = sum(1 for px, py in placed
                   if abs(px - x) < 0.004 and abs(py - y) < span * 0.03)
        ax.annotate(C.COND_LABEL[cond].replace("\\_", "_"), (x, y), fontsize=7,
                    xytext=(4, 3 - 9 * drop), textcoords="offset points")
        placed.append((x, y))
    ax.set_xlabel("label agreement")
    ax.set_ylabel("non-collapsed GF (%)")
    ax.set_title(f"T{test[-1]}  (r={r:+.2f}, p={p:.2f}, {len(keep)}/{len(frame)} on the frontier)")
    ax.legend(frameon=False, fontsize=7, loc="best")
fig.suptitle("Fairness--utility frontier within each regime", y=1.02)
fig.tight_layout()
fig.savefig(FIGURES / "pareto_frontier.png", bbox_inches="tight")
plt.show()

## 10. Export

The tables above are written to `analysis/` as CSV. The LaTeX the manuscript
consumes is produced from the same `cells.py`, so the two paths cannot diverge.

In [ ]:
OUTDIR = Path("analysis")
exports = {
    "cells_full.csv": flat,
    "audit_levels.csv": levels,
    "sensitivity.csv": sensitivity.reset_index(),
    "bootstrap_gf.csv": boot,
}
if DUAL_AVAILABLE:
    exports["transfer.csv"] = transfer.reset_index()
    exports["delta_gf.csv"] = deltas.reset_index()
if base is not None:
    exports["baselines.csv"] = base

for name, frame in exports.items():
    frame.to_csv(OUTDIR / name, index=name in {"audit_levels.csv"})
    print(f"wrote {OUTDIR / name}  ({len(frame)} rows)")

In [ ]:
print("Summary")
print("=" * 60)
for test in sorted(flat["test"].unique()):
    part = flat[flat["test"] == test]
    print(f"T{test[-1]}: {len(part):>3} cells | raw GF {int(part['raw_gf'].sum()):>3} | "
          f"non-collapsed {int(part['gf'].sum()):>3} | useful {int(part['gf_useful'].sum()):>3}")